# 셋팅 #

필요한 설정을 위해 이 셀을 실행하세요!


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')
ryanholbrook_ts_course_data_path = kagglehub.dataset_download('ryanholbrook/ts-course-data')

print('Data source import complete.')


In [ ]:
import os
from pathlib import Path

# Define the target directory where learntools expects to find the data
input_base_path = Path('/input')
input_base_path.mkdir(parents=True, exist_ok=True)

# Create a symbolic link for the store-sales-time-series-forecasting competition data
symlink_target_comp = input_base_path / 'store-sales-time-series-forecasting'
symlink_source_comp = store_sales_time_series_forecasting_path

if symlink_target_comp.exists() or symlink_target_comp.is_symlink():
    if symlink_target_comp.is_symlink():
        os.unlink(symlink_target_comp)
    else:
        # If it's a directory, remove it to replace with symlink
        import shutil
        shutil.rmtree(symlink_target_comp)

os.symlink(symlink_source_comp, symlink_target_comp)
print(f"Created symlink: {symlink_target_comp} -> {symlink_source_comp}")

# Create a symbolic link for the ryanholbrook/ts-course-data dataset
symlink_target_data = input_base_path / 'ts-course-data'
symlink_source_data = ryanholbrook_ts_course_data_path

if symlink_target_data.exists() or symlink_target_data.is_symlink():
    if symlink_target_data.is_symlink():
        os.unlink(symlink_target_data)
    else:
        import shutil
        shutil.rmtree(symlink_target_data)

os.symlink(symlink_source_data, symlink_target_data)
print(f"Created symlink: {symlink_target_data} -> {symlink_source_data}")


In [ ]:
!git clone https://github.com/Kaggle/learntools.git
!mv learntools learntools_dir
!mv learntools_dir/learntools learntools

# 계절성(Seasonality)이란? #

시계열의 평균이 규칙적으로 반복되는 변화를 보이면 그 시계열은 **계절성(seasonality)**을 가진다고 말합니다. 계절 변화는 대개 시계와 달력의 리듬을 따르므로 하루, 일주일, 일년처럼 일정한 주기로 반복됩니다. 계절성은 하루·일년 단위의 자연 주기나 특정 날짜와 시간을 둘러싼 사회적 관습에서 비롯되는 경우가 많습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/ViYbSxS.png" width=800, alt="">
<figcaption style="textalign: center; font-style: italic"><center>네 개의 시계열에서 나타나는 계절 패턴.
</center></figcaption>
</figure>

이번 강의에서는 계절성을 모델링하는 두 종류의 피처를 배웁니다. 첫 번째인 지표(indicator)는 일 단위 관측에서 주간 계절성을 다루는 것처럼 관측 횟수가 비교적 적은 계절에 잘 맞습니다. 두 번째인 푸리에(Fourier) 피처는 일 단위 관측에서 연간 계절성을 다루는 것처럼 관측 수가 많은 계절을 다루기에 좋습니다.

# 계절 플롯과 계절 지표 #

시리즈의 추세를 파악할 때 이동 평균 플롯을 사용했던 것처럼, 계절 패턴을 찾을 때는 **계절 플롯(seasonal plot)**을 사용할 수 있습니다.

계절 플롯은 시계열을 공통된 주기, 즉 살펴보고 싶은 "계절"에 맞춰 여러 구간으로 나누어 그립니다. 아래 그림은 위키백과 *Trigonometry* 문서의 일일 조회수를 *주간* 주기에 맞춰 그린 계절 플롯입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/bd7D4NJ.png" width=800, alt="">
<figcaption style="textalign: center; font-style: italic"><center>평일에는 높고 주말에는 낮아지는 뚜렷한 주간 계절 패턴이 보입니다.
</center></figcaption>
</figure>

### 계절 지표

**계절 지표**는 시계열이 계절 주기 안에서 어느 위치에 있는지를 나타내는 이진 피처입니다. 계절 주기를 범주형 피처로 보고 원-핫 인코딩을 적용하면 계절 지표를 얻을 수 있습니다.

요일을 원-핫 인코딩하면 주간 계절 지표가 만들어집니다. *Trigonometry* 시리즈에 주간 지표를 만들면 새로운 "더미" 피처가 여섯 개 추가됩니다. (선형 회귀는 지표 중 하나를 제거했을 때 가장 잘 작동합니다. 아래 표에서는 월요일을 제거했습니다.)

| Date       | Tuesday | Wednesday | Thursday | Friday | Saturday | Sunday |
|------------|---------|-----------|----------|--------|----------|--------|
| 2016-01-04 | 0.0     | 0.0       | 0.0      | 0.0    | 0.0      | 0.0    |
| 2016-01-05 | 1.0     | 0.0       | 0.0      | 0.0    | 0.0      | 0.0    |
| 2016-01-06 | 0.0     | 1.0       | 0.0      | 0.0    | 0.0      | 0.0    |
| 2016-01-07 | 0.0     | 0.0       | 1.0      | 0.0    | 0.0      | 0.0    |
| 2016-01-08 | 0.0     | 0.0       | 0.0      | 1.0    | 0.0      | 0.0    |
| 2016-01-09 | 0.0     | 0.0       | 0.0      | 0.0    | 1.0      | 0.0    |
| 2016-01-10 | 0.0     | 0.0       | 0.0      | 0.0    | 0.0      | 1.0    |
| 2016-01-11 | 0.0     | 0.0       | 0.0      | 0.0    | 0.0      | 0.0    |
| ...        | ...     | ...       | ...      | ...    | ...      | ...    |


계절 지표를 학습 데이터에 추가하면 모델이 계절 주기 안에서 평균이 어떻게 달라지는지 구분할 수 있습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/hIlF5j5.png" width=800, alt="">
<figcaption style="textalign: center; font-style: italic"><center>일반 선형 회귀는 계절 안 각 시점의 평균을 학습합니다.</center></figcaption>
</figure>

지표는 스위치처럼 동작합니다. 어느 시점이든 값이 `1`(켬)이 되는 지표는 최대 하나뿐입니다. 선형 회귀는 `Mon`의 기준값 `2379`를 학습한 뒤, 그날 켜진 지표의 값만큼을 더하거나 빼서 예측을 조정합니다. 나머지 지표는 `0`이라서 사라집니다.

# 푸리에 피처와 주파수 스펙트럼(Periodogram) #

지금 설명할 피처는 관측 수가 많은 긴 계절에 더 적합합니다. 날짜마다 피처를 하나씩 만드는 대신, 계절 곡선의 전체 모양을 소수의 피처로 포착하려는 방식입니다.

*Trigonometry* 시리즈의 연간 계절 플롯을 보면 여러 주파수가 반복되는 것을 볼 수 있습니다. 1년에 세 번 나타나는 완만한 오르내림, 1년에 52번 반복되는 짧은 주간 움직임 등입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/NJcaEdI.png" width=800, alt="">
<figcaption style="textalign: center; font-style: italic"><center><em>Wiki Trigonometry</em> 시리즈의 연간 계절성.</center></figcaption>
</figure>

푸리에 피처는 이런 주파수를 포착하려는 시도입니다. 우리가 모델링하려는 계절과 동일한 주파수를 가진 주기 함수를 학습 데이터에 포함시키는 것이죠. 여기서 사용하는 곡선은 삼각함수인 사인과 코사인입니다.

**푸리에 피처(Fourier features)**는 사인·코사인 한 쌍으로 구성되며, 계절 안에서 가능한 주파수마다 한 쌍씩 추가합니다. 연간 계절이라면 1년에 한 번, 두 번, 세 번 ... 과 같은 주파수가 됩니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/bKOjdU7.png" width=600, alt="A top figure and a bottom figure, each showing a sine curve and a cosine curve. The curves in the top plot both have frequency of once per year, while the curves in the bottom plot both have a frequency of twice per year.">
<figcaption style="textalign: center; font-style: italic"><center>연간 계절성을 위한 첫 번째와 두 번째 푸리에 쌍. <strong>위:</strong> 1년에 한 번 반복. <strong>아래:</strong> 1년에 두 번 반복.</center></figcaption>
</figure>

이렇게 만든 사인/코사인 곡선을 학습 데이터에 추가하면 선형 회귀가 계수를 학습해 타깃 시리즈의 계절 성분을 맞춥니다. 아래 그림은 선형 회귀가 네 개의 푸리에 쌍을 이용해 <em>Wiki Trigonometry</em> 시리즈의 연간 계절성을 모델링한 모습입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/mijPhko.png" width=600, alt="">
<figcaption style="textalign: center; font-style: italic"><center><strong>위:</strong> 네 개의 푸리에 쌍(각기 다른 주파수의 사인과 코사인) 곡선. <strong>아래:</strong> 이 곡선들을 합한 결과가 계절 패턴을 근사합니다.</center></figcaption>
</figure>

연간 계절성을 근사하는 데 단 여덟 개의 피처(네 쌍의 사인/코사인)만 필요했다는 사실에 주목하세요. 계절 지표를 사용했다면 1년의 각 날짜마다 피처가 필요했으니 수백 개가 필요했을 겁니다. 푸리에 피처로 계절성의 “주요 효과”만 모델링하면 훨씬 적은 피처로도 충분해 계산 비용을 줄이고 과적합 위험도 낮출 수 있습니다.

### 주파수 스펙트럼(Periodogram)으로 푸리에 피처 선택하기

그렇다면 실제로 몇 개의 푸리에 쌍을 써야 할까요? 이때 **주파수 스펙트럼(periodogram)**이 도움이 됩니다. 주파수 스펙트럼은 시계열에 어떤 주파수가 얼마나 강하게 존재하는지 알려 줍니다. 그래프 y축의 값은 각 주파수의 사인·코사인 계수를 각각 `a`, `b`라고 했을 때 `(a ** 2 + b ** 2) / 2`입니다(위의 *Fourier Components* 그림에서처럼).

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/PK6WEe3.png" width=600, alt="">
<figcaption style="textalign: center; font-style: italic"><center><em>Wiki Trigonometry</em> 시리즈의 주파수 스펙트럼.</center></figcaption>
</figure>

왼쪽에서 오른쪽으로 보면 스펙트럼이 <em>Quarterly</em>(1년에 네 번) 이후 크게 감소합니다. 그래서 연간 계절성을 모델링할 때 네 쌍의 푸리에 피처만 사용했습니다. <em>Weekly</em> 주파수는 지표로 더 잘 모델링되므로 여기서는 제외했습니다.

### 푸리에 피처 계산(선택 사항)

푸리에 피처가 어떻게 계산되는지 아는 것이 필수는 아니지만, 자세한 과정을 보면 이해에 도움이 될 수 있습니다. 아래 숨겨진 셀에서는 시계열의 인덱스로부터 푸리에 피처 집합을 구성하는 방법을 보여 줍니다. (실제로는 `statsmodels` 라이브러리의 함수로 더 간단하게 생성할 것입니다.)


In [ ]:
import numpy as np


def fourier_features(index, freq, order):
    time = np.arange(len(index), dtype=np.float32)
    k = 2 * np.pi * (1 / freq) * time
    features = {}
    for i in range(1, order + 1):
        features.update({
            f"sin_{freq}_{i}": np.sin(i * k),
            f"cos_{freq}_{i}": np.cos(i * k),
        })
    return pd.DataFrame(features, index=index)


# y가 일 단위 관측을 갖고 연간 계절성을 지닌다고 가정하면,
# 4차(새로운 피처 8개)까지의 푸리에 피처를 계산하는 예시는 다음과 같습니다:
#
# fourier_features(y, freq=365.25, order=4) y가 아직 없으니 실행되지는 않습니다.


# 예제 - 터널 교통량 #

이번에도 *Tunnel Traffic* 데이터셋을 사용합니다. 아래 숨겨진 셀은 데이터를 불러오고 `seasonal_plot`, `plot_periodogram` 함수를 정의합니다.


In [ ]:
from pathlib import Path
from warnings import simplefilter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

simplefilter("ignore")  # 경고를 무시해 출력 셀을 깔끔하게 유지

# Matplotlib 기본 설정
sns.set_style("whitegrid") # Fix: Use sns.set_style for seaborn styles
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=16,
    titlepad=10,
)
plot_params = dict(
    color="0.75",
    style=".-",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
    legend=False,
)
%config InlineBackend.figure_format = 'retina'


# 주석 참고: https://stackoverflow.com/a/49238256/5769929
def seasonal_plot(X, y, period, freq, ax=None):
    if ax is None:
        _, ax = plt.subplots()
    palette = sns.color_palette("husl", n_colors=X[period].nunique(),)
    ax = sns.lineplot(
        x=freq,
        y=y,
        hue=period,
        data=X,
        ci=False,
        ax=ax,
        palette=palette,
        legend=False,
    )
    ax.set_title(f"Seasonal Plot ({period}/{freq})")
    for line, name in zip(ax.lines, X[period].unique()):
        y_ = line.get_ydata()[-1]
        ax.annotate(
            name,
            xy=(1, y_),
            xytext=(6, 0),
            color=line.get_color(),
            xycoords=ax.get_yaxis_transform(),
            textcoords="offset points",
            size=14,
            va="center",
        )
    return ax


def plot_periodogram(ts, detrend='linear', ax=None):
    from scipy.signal import periodogram
    fs = pd.Timedelta("365D") / pd.Timedelta("1D")
    freqencies, spectrum = periodogram(
        ts,
        fs=fs,
        detrend=detrend,
        window="boxcar",
        scaling='spectrum',
    )
    if ax is None:
        _, ax = plt.subplots()
    ax.step(freqencies, spectrum, color="purple")
    ax.set_xscale("log")
    ax.set_xticks([1, 2, 4, 6, 12, 26, 52, 104])
    ax.set_xticklabels(
        [
            "Annual (1)",
            "Semiannual (2)",
            "Quarterly (4)",
            "Bimonthly (6)",
            "Monthly (12)",
            "Biweekly (26)",
            "Weekly (52)",
            "Semiweekly (104)",
        ],
        rotation=30,
    )
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.set_ylabel("Variance")
    ax.set_title("Periodogram")
    return ax


data_dir = Path("../input/ts-course-data")
tunnel = pd.read_csv(data_dir / "tunnel.csv", parse_dates=["Day"])
tunnel = tunnel.set_index("Day").to_period("D")

일주일과 일년 단위의 계절 플롯을 살펴보겠습니다.


In [ ]:
X = tunnel.copy()

# 일주일 안의 요일
X["day"] = X.index.dayofweek  # x축(주파수)
X["week"] = X.index.week  # 계절 주기(period)

# 일년 안의 날짜
X["dayofyear"] = X.index.dayofyear
X["year"] = X.index.year
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(11, 12))
seasonal_plot(X, y="NumVehicles", period="week", freq="day", ax=ax0)
seasonal_plot(X, y="NumVehicles", period="year", freq="dayofyear", ax=ax1);


이번에는 주파수 스펙트럼을 확인해 봅시다:


In [ ]:
plot_periodogram(tunnel.NumVehicles);

주파수 스펙트럼은 앞의 계절 플롯과 같은 결론을 보여 줍니다. 주간 계절성이 강하고 연간 계절성은 조금 더 약합니다. 주간 계절성은 지표로, 연간 계절성은 푸리에 피처로 모델링하겠습니다. 오른쪽에서 왼쪽으로 보면 스펙트럼이 <em>Bimonthly (6)</em>와 <em>Monthly (12)</em> 사이에서 크게 줄어드니 푸리에 쌍은 10개 정도 사용해 보죠.

계절 피처는 2강에서 추세 피처를 만들 때 사용했던 `DeterministicProcess`로 생성하겠습니다. 두 종류의 계절 주기(주간과 연간)를 모두 쓰려면 하나를 "추가 항(additional term)"으로 지정하면 됩니다.


In [ ]:
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

fourier = CalendarFourier(freq="A", order=10)  # "A"nnual 계절성을 위한 sin/cos 쌍 10개

dp = DeterministicProcess(
    index=tunnel.index,
    constant=True,               # 편향(bias, y절편)을 위한 더미 피처
    order=1,                     # 추세(order 1은 선형)
    seasonal=True,               # 주간 계절성(지표)
    additional_terms=[fourier],  # 연간 계절성(푸리에)
    drop=True,                   # 다중공선성을 피하기 위해 항 제거
)

X = dp.in_sample()  # tunnel.index에 있는 날짜용 피처 생성


피처 세트를 만들었으니 모델을 학습하고 예측을 생성할 준비가 되었습니다. 훈련 데이터를 넘어서는 90일 예측을 추가해 모델이 관측 기간 밖에서 어떻게 외삽하는지 살펴보겠습니다. 코드는 앞선 강의들과 동일합니다.


In [ ]:

y = tunnel["NumVehicles"]


model = LinearRegression(fit_intercept=False)
_ = model.fit(X, y) # 올바른 피처로 모델 학습

y_pred = pd.Series(model.predict(X), index=y.index) # 학습 피처로 예측
X_fore = dp.out_of_sample(steps=90)
y_fore = pd.Series(model.predict(X_fore), index=X_fore.index)

ax = y.plot(color='0.25', style='.', title="Tunnel Traffic - Seasonal Forecast")
ax = y_pred.plot(ax=ax, label="Seasonal")
ax = y_fore.plot(ax=ax, label="Seasonal Forecast", color='C3')
_ = ax.legend()

---

아직 예측을 개선할 여지가 남아 있습니다. 다음 강의에서는 시계열 자체를 피처로 사용하는 방법을 배워 사이클(cycle)과 같은 또 다른 성분을 모델링해 보겠습니다.

# 연습 문제를 풀어볼까요? #



# 셋팅 #

환경을 모두 준비하려면 이 셀을 실행하세요!

In [ ]:
# 피드백 시스템 설정
from learntools.core import binder
binder.bind(globals())
from learntools.time_series.ex3 import *

# 노트북 환경 설정
from pathlib import Path
from learntools.time_series.style import *  # 플롯 스타일 설정
from learntools.time_series.utils import plot_periodogram, seasonal_plot

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess


comp_dir = Path('../input/store-sales-time-series-forecasting')

holidays_events = pd.read_csv(
    comp_dir / "holidays_events.csv",
    dtype={
        'type': 'category',
        'locale': 'category',
        'locale_name': 'category',
        'description': 'category',
        'transferred': 'bool',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
holidays_events = holidays_events.set_index('date').to_period('D')

store_sales = pd.read_csv(
    comp_dir / 'train.csv',
    usecols=['store_nbr', 'family', 'date', 'sales'],
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'sales': 'float32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
store_sales['date'] = store_sales.date.dt.to_period('D')
store_sales = store_sales.set_index(['store_nbr', 'family', 'date']).sort_index()
average_sales = (
    store_sales
    .groupby('date').mean()
    .squeeze()
    .loc['2017']
)


-------------------------------------------------------------------------------

다음 계절 플롯을 살펴보세요:


In [ ]:
X = average_sales.to_frame()
X["week"] = X.index.week
X["day"] = X.index.dayofweek
seasonal_plot(X, y='sales', period='week', freq='day');

그리고 주파수 스펙트럼도 확인해 보세요:


In [ ]:
plot_periodogram(average_sales);

# 1) 계절성 판별하기

어떤 종류의 계절성이 있다고 생각되나요? 생각을 정리한 뒤 다음 셀을 실행해 해설을 확인하세요.


In [ ]:
# 해설 보기(이 셀을 실행해야 점수를 받을 수 있습니다!)
q_1.check()


-------------------------------------------------------------------------------

# 2) 계절 피처 만들기

`DeterministicProcess`와 `CalendarFourier`를 사용해 다음을 생성하세요.
- 주간 계절 지표
- 월간 계절을 위한 4차 푸리에 피처


In [ ]:
y = average_sales.copy()

# 여기에 코드를 작성하세요
fourier = ____
dp = DeterministicProcess(
    index=y.index,
    constant=True,
    order=1,
    # 여기에 코드를 작성하세요
    # ____
    drop=True,
)
X = ____

# 답 확인하기
q_2.check()


In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_2.hint()
#q_2.solution()


문제를 풀어야 실행이 가능합니다.
이제 이 셀을 실행해 계절 모델을 학습하세요.


In [ ]:
model = LinearRegression().fit(X, y)
y_pred = pd.Series(
    model.predict(X),
    index=X.index,
    name='Fitted',
)

y_pred = pd.Series(model.predict(X), index=X.index)
ax = y.plot(**plot_params, alpha=0.5, title="Average Sales", ylabel="items sold")
ax = y_pred.plot(ax=ax, label="Seasonal")
ax.legend();

시리즈에서 추세나 계절을 제거하는 작업을 각각 **디트렌딩(detrending)**, **디시즌얼라이징(deseasonalizing)**이라고 부릅니다.

계절 성분을 제거한 시리즈의 주파수 스펙트럼을 살펴보세요.


In [ ]:
y_deseason = y - y_pred

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(10, 7))
ax1 = plot_periodogram(y, ax=ax1)
ax1.set_title("Product Sales Frequency Components")
ax2 = plot_periodogram(y_deseason, ax=ax2);
ax2.set_title("Deseasonalized");

# 3) 남은 계절성 확인하기

이 주파수 스펙트럼을 보면 모델이 *Average Sales*의 계절성을 얼마나 잘 포착했는지 짐작할 수 있나요? 주파수 스펙트럼이 deseasonalized 시리즈의 시계열 그래프와 일치하나요?


In [ ]:
# 해설 보기(이 셀을 실행해야 점수를 받을 수 있습니다!)
q_3.check()


-------------------------------------------------------------------------------

*Store Sales* 데이터셋에는 에콰도르의 공휴일 정보가 들어 있습니다.


In [ ]:
# 학습 데이터에 있는 국가 공휴일과 지역 공휴일
holidays = (
    holidays_events
    .query("locale in ['National', 'Regional']")
    .loc['2017':'2017-08-15', ['description']]
    .assign(description=lambda x: x.description.cat.remove_unused_categories())
)

display(holidays)


디시즌얼라이즈된 *Average Sales*를 보면 이러한 공휴일이 예측에 도움이 될 수도 있을 것 같습니다.


In [ ]:
ax = y_deseason.plot(**plot_params)
plt.plot_date(holidays.index, y_deseason[holidays.index], color='C3')
ax.set_title('National and Regional Holidays');

# 4) 공휴일 피처 만들기

이 정보를 모델이 활용하도록 하려면 어떤 피처를 만들면 좋을까요? 다음 셀에서 답을 코드로 작성해 보세요. (Scikit-learn과 Pandas에는 이 작업을 쉽게 도와주는 도구가 있습니다. 자세한 내용이 필요하면 `hint`를 참고하세요.)



In [ ]:
# 여기에 코드를 작성하세요
X_holidays = ____

X2 = X.join(X_holidays, on='date').fillna(0.0)



In [ ]:
# 아래 줄들은 힌트나 해설 코드를 제공합니다

#q_4.hint()
#q_4.hint(2)
#q_4.solution()
# ohe = OneHotEncoder(sparse_output=False) 원핫 인코더 생성하는 코드를 바꾸셔야 동작합니다.


이 셀을 사용해 공휴일 피처를 추가한 계절 모델을 학습하세요. 예측값이 더 좋아졌나요?


In [ ]:
model = LinearRegression().fit(X2, y)
y_pred = pd.Series(
    model.predict(X2),
    index=X2.index,
    name='Fitted',
)

y_pred = pd.Series(model.predict(X2), index=X2.index)
ax = y.plot(**plot_params, alpha=0.5, title="Average Sales", ylabel="items sold")
ax = y_pred.plot(ax=ax, label="Seasonal")
ax.legend();

In [ ]:
y = store_sales.unstack(['store_nbr', 'family']).loc["2017"]

# 학습 데이터 생성
fourier = CalendarFourier(freq='M', order=4)
dp = DeterministicProcess(
    index=y.index,
    constant=True,
    order=1,
    seasonal=True,
    additional_terms=[fourier],
    drop=True,
)
X = dp.in_sample()
X['NewYear'] = (X.index.dayofyear == 1)

model = LinearRegression(fit_intercept=False)
model.fit(X, y)
y_pred = pd.DataFrame(model.predict(X), index=X.index, columns=y.columns)


이 셀을 실행하면 해당 모델이 만든 일부 예측을 확인할 수 있습니다.


In [ ]:
STORE_NBR = '1'  # 1 - 54
FAMILY = 'PRODUCE'
# 제품군 목록을 보려면 아래 줄의 주석을 해제하세요
# display(store_sales.index.get_level_values('family').unique())

ax = y.loc(axis=1)['sales', STORE_NBR, FAMILY].plot(**plot_params)
ax = y_pred.loc(axis=1)['sales', STORE_NBR, FAMILY].plot(ax=ax)
ax.set_title(f'{FAMILY} Sales at Store {STORE_NBR}');


마지막으로, 이 셀은 테스트 데이터를 불러오고 예측 기간에 대한 피처를 만든 다음 `submission.csv` 제출 파일을 생성합니다.


In [ ]:
df_test = pd.read_csv(
    comp_dir / 'test.csv',
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
df_test['date'] = df_test.date.dt.to_period('D')
df_test = df_test.set_index(['store_nbr', 'family', 'date']).sort_index()

# 테스트 세트를 위한 피처 생성
X_test = dp.out_of_sample(steps=16)
X_test.index.name = 'date'
X_test['NewYear'] = (X_test.index.dayofyear == 1)

y_submit = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
y_submit = y_submit.stack(['store_nbr', 'family'])
y_submit = y_submit.join(df_test.id).reindex(columns=['id', 'sales'])
y_submit.to_csv('./submission.csv', index=False)


Store Sales 제출 해보셔도 좋습니다 :)